<a href="https://colab.research.google.com/github/lokeshp051/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokeshp051/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
BASE = "hf://datasets/FlyRank/internship-warehouse"

In [ ]:
import pandas as pd
import numpy as np

full = con.sql(f"""
WITH daily AS (
  SELECT report_date, client_hash_id, content_hash_id,
         gsc_impressions, gsc_clicks, gsc_sum_position
  FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
  WHERE gsc_data_available IS TRUE
),
h1 AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impressions_h1,
         SUM(gsc_clicks) AS clicks_h1,
         SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions),0) AS avg_position_h1,
         SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions),0) AS ctr_h1
  FROM daily WHERE report_date <= DATE '2026-03-15'
  GROUP BY 1,2
),
h2 AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_h2
  FROM daily WHERE report_date > DATE '2026-03-15'
  GROUP BY 1,2
)
SELECT h1.*, d.word_count, h2.impressions_h2,
       (h2.impressions_h2 < h1.impressions_h1) AS is_declining_label
FROM h1
JOIN h2 USING (client_hash_id, content_hash_id)
LEFT JOIN read_parquet('{BASE}/dim_content.parquet') d USING (content_hash_id)
WHERE h1.impressions_h1 > 20
""").df()

full["is_declining_label"] = full["is_declining_label"].astype(int)
full.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(108019, 9)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

features = ["impressions_h1", "clicks_h1", "avg_position_h1", "ctr_h1", "word_count"]
X = full[features].fillna(0)
y = full["is_declining_label"]
groups = full["client_hash_id"]

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))

tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(X.iloc[train_idx], y.iloc[train_idx])

full["decline_score"] = tree.predict_proba(X)[:, 1]

def reason_code(row):
    if row["decline_score"] > 0.7 and row["ctr_h1"] < 0.002:
        return "HIGH_RISK_LOW_CTR"
    elif row["decline_score"] > 0.7:
        return "HIGH_RISK_GENERAL"
    elif row["avg_position_h1"] <= 10 and row["ctr_h1"] < 0.002:
        return "LOW_CTR_FOR_POSITION"
    else:
        return "MONITOR"

full["reason_code"] = full.apply(reason_code, axis=1)

def action_label(rc):
    return {
        "HIGH_RISK_LOW_CTR": "urgent_refresh_and_snippet_rewrite",
        "HIGH_RISK_GENERAL": "schedule_content_refresh",
        "LOW_CTR_FOR_POSITION": "review_title_and_snippet",
        "MONITOR": "no_action_monitor_only"
    }[rc]

full["action_label"] = full["reason_code"].apply(action_label)

queue = full[full["reason_code"] != "MONITOR"].sort_values("decline_score", ascending=False)
print("Actionable queue size:", len(queue))
queue[["client_hash_id","content_hash_id","decline_score","reason_code","action_label"]].head(10)

Actionable queue size: 38813


,client_hash_id,content_hash_id,decline_score,reason_code,action_label
14213,client_23a62021009f63c4,content_48211892d42d0f22,0.809627,HIGH_RISK_LOW_CTR,urgent_refresh_and_snippet_rewrite
14212,client_23a62021009f63c4,content_481fc13c36c5fa4b,0.809627,HIGH_RISK_LOW_CTR,urgent_refresh_and_snippet_rewrite
14194,client_23a62021009f63c4,content_47a657be9bc46cb6,0.809627,HIGH_RISK_LOW_CTR,urgent_refresh_and_snippet_rewrite
14192,client_23a62021009f63c4,content_47a1a667503bd92e,0.809627,HIGH_RISK_LOW_CTR,urgent_refresh_and_snippet_rewrite
14190,client_23a62021009f63c4,content_4788c4ed0cd3a7df,0.809627,HIGH_RISK_LOW_CTR,urgent_refresh_and_snippet_rewrite
14271,client_23a62021009f63c4,content_4902e4bf14bb6d28,0.809627,HIGH_RISK_LOW_CTR,urgent_refresh_and_snippet_rewrite
14260,client_23a62021009f63c4,content_48e429fec313253b,0.809627,HIGH_RISK_LOW_CTR,urgent_refresh_and_snippet_rewrite
82792,client_23a62021009f63c4,content_dda8be96d809ab55,0.809627,HIGH_RISK_LOW_CTR,urgent_refresh_and_snippet_rewrite
82706,client_23a62021009f63c4,content_db9e902a9fb14e10,0.809627,HIGH_RISK_LOW_CTR,urgent_refresh_and_snippet_rewrite
82727,client_23a62021009f63c4,content_dc169b6dcd625110,0.809627,HIGH_RISK_LOW_CTR,urgent_refresh_and_snippet_rewrite


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue ranks 38,813 pages by decline_score (from the Week-5 decision tree) into three reason codes: HIGH_RISK_LOW_CTR (high decline risk + CTR below 0.002 → urgent_refresh_and_snippet_rewrite), HIGH_RISK_GENERAL (high decline risk, CTR not the driver → schedule_content_refresh), and LOW_CTR_FOR_POSITION (well-positioned but under-clicking → review_title_and_snippet). Pages that don't trip any threshold get MONITOR / no_action_monitor_only and are excluded from the actionable queue. This mirrors the same reason-code + action-label pattern from the Week-4 baseline and Week-5 session, now driven by the trained model's score instead of a single hand-rule.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use: this queue is decision-support for a human content reviewer prioritizing which pages to look at first — not an automated content-editing pipeline. Limits: trained on one month (March 2026) of one dataset slice; decline_score reflects observed patterns in this portfolio, not a guarantee for any individual page; the label itself is a proxy (impressions dropped h1→h2), not a verified business outcome; word_count carries surprisingly high feature importance (Week 5 finding) which may reflect a correlated content-type effect rather than a causal lever.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human review required for every flagged page before any action is taken — this queue surfaces candidates, it does not approve edits. No-go / should NOT be automated: auto-publishing rewritten titles/snippets without editorial review; auto-deleting or de-indexing "MONITOR" pages; treating decline_score as a ranking-algorithm prediction (it isn't — it's a portfolio-internal proxy); applying this model to a client/vertical outside the training data without re-validation; using this queue for any content the client hasn't authorized for automated analysis.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Monitor: track actual impression change 30/60 days after any action taken on a flagged page, compared to a matched unflagged control. Retrain triggers: if precision@50 on a fresh month drops meaningfully below the Week-5 baseline (0.64), if the reason-code distribution shifts sharply (e.g. HIGH_RISK_LOW_CTR balloons past historical share), or every quarter regardless, since search behavior and the underlying warehouse data change over time.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
import os
import json

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

queue[["client_hash_id","content_hash_id","decline_score","reason_code","action_label","impressions_h1","clicks_h1","avg_position_h1","ctr_h1","word_count"]].to_csv("work/outputs/action_playbook_queue.csv", index=False)

metrics = {"total_pages_scored": len(full), "actionable_queue_size": len(queue), "reason_code_counts": full["reason_code"].value_counts().to_dict(), "model": "DecisionTreeClassifier(max_depth=4)", "split": "grouped_by_client", "precision_at_50_grouped_split": 0.600}

with open("work/outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved CSV rows:", len(queue))
print(json.dumps(metrics, indent=2))

Saved CSV rows: 38813
{
  "total_pages_scored": 108019,
  "actionable_queue_size": 38813,
  "reason_code_counts": {
    "MONITOR": 69206,
    "LOW_CTR_FOR_POSITION": 35925,
    "HIGH_RISK_LOW_CTR": 2422,
    "HIGH_RISK_GENERAL": 466
  },
  "model": "DecisionTreeClassifier(max_depth=4)",
  "split": "grouped_by_client",
  "precision_at_50_grouped_split": 0.6
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.